# P&C Claims Reserving Walkthrough

A step-by-step run through the reserving engine in `src/reserving.py`: build the
triangle, derive development factors, select a tail, project ultimates, decompose the
reserve into case reserves and IBNR, and quantify uncertainty with Mack (1993).

**The data is synthetic and illustrative.** It is deliberately smooth, which matters
when reading the uncertainty numbers at the end.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.reserving import (
    build_loss_triangle, development_factors, cumulative_factors,
    estimate_tail_factor, mack_standard_error, project_reserves,
)

df = pd.read_csv('../data/pc_claims_dataset.csv')
print(f"{len(df)} rows, {df['line_of_business'].nunique()} lines of business")
df.head()

30 rows, 2 lines of business


,line_of_business,accident_year,development_year,paid_claims,incurred_claims,earned_premium,claim_count
0,Commercial Property,2019,0,1250000,2100000,5000000,450
1,Commercial Property,2019,1,2300000,2850000,5000000,450
2,Commercial Property,2019,2,3100000,3400000,5000000,450
3,Commercial Property,2019,3,3550000,3700000,5000000,450
4,Commercial Property,2019,4,3750000,3800000,5000000,450


## 1. The triangle

Cumulative paid claims by accident year and development year. The staircase shape is
the point: recent accident years have had fewer periods to develop.

In [2]:
paid = build_loss_triangle(df, 'Commercial Property', 'paid_claims')
incurred = build_loss_triangle(df, 'Commercial Property', 'incurred_claims')
paid

development_year,0,1,2,3,4
accident_year,,,,,
2019,1250000.0,2300000.0,3100000.0,3550000.0,3750000.0
2020,1400000.0,2550000.0,3450000.0,3900000.0,NaN
2021,1550000.0,2800000.0,3800000.0,NaN,NaN
2022,1700000.0,3050000.0,NaN,NaN,NaN
2023,1850000.0,NaN,NaN,NaN,NaN


Incurred sits above paid. The difference is the **case reserve**: what claim handlers
expect still to be paid on claims they already know about.

In [3]:
(incurred - paid).rename_axis(columns='development_year')

development_year,0,1,2,3,4
accident_year,,,,,
2019,850000.0,550000.0,300000.0,150000.0,50000.0
2020,900000.0,550000.0,300000.0,150000.0,NaN
2021,950000.0,600000.0,350000.0,NaN,NaN
2022,1000000.0,650000.0,NaN,NaN,NaN
2023,1100000.0,NaN,NaN,NaN,NaN


## 2. Development factors

Volume-weighted age-to-age factors, using only the accident years where both
development periods are observed:

$$f_k = \frac{\sum_i C_{i,k+1}}{\sum_i C_{i,k}}$$

In [4]:
factors = development_factors(paid)
for k, f in sorted(factors.items()):
    print(f"  DY {k} -> {k+1}:  {f:.6f}")

# Hand-check the first factor against the triangle.
hand = (2300000 + 2550000 + 2800000 + 3050000) / (1250000 + 1400000 + 1550000 + 1700000)
print(f"\nhand-computed f_0 = {hand:.6f}   matches: {abs(hand - factors[0]) < 1e-12}")

  DY 0 -> 1:  1.813559
  DY 1 -> 2:  1.352941
  DY 2 -> 3:  1.137405
  DY 3 -> 4:  1.056338

hand-computed f_0 = 1.813559   matches: True


Every paid factor is at or above 1.0. Cumulative paid claims cannot decrease, so a
paid link ratio below 1.0 is always a construction error rather than a finding.

## 3. Tail factor

The triangle stops at development year 4, but that does not mean development stops.
Rather than assume a tail of 1.0, derive it from the oldest accident year's open case
reserve at its final observed development year.

In [5]:
tail = estimate_tail_factor(paid, incurred)
print(f"Commercial Property derived tail: {tail:.6f}")
print(f"  AY2019 at DY4: paid {paid.loc[2019, 4]:,.0f}, incurred {incurred.loc[2019, 4]:,.0f}")
print(f"  still open: {incurred.loc[2019, 4] - paid.loc[2019, 4]:,.0f}")

motor_paid = build_loss_triangle(df, 'Private Motor', 'paid_claims')
motor_inc = build_loss_triangle(df, 'Private Motor', 'incurred_claims')
print(f"\nPrivate Motor derived tail: {estimate_tail_factor(motor_paid, motor_inc):.6f}"
      f"  (fully run off, so no tail is warranted)")

Commercial Property derived tail: 1.013333
  AY2019 at DY4: paid 3,750,000, incurred 3,800,000
  still open: 50,000

Private Motor derived tail: 1.000000  (fully run off, so no tail is warranted)


## 4. Projection and reserve decomposition

Three different numbers, which an earlier version of this repo conflated:

| Quantity | Definition |
|---|---|
| Total unpaid | Ultimate less cumulative paid |
| Case reserves | Incurred less cumulative paid |
| IBNR | Ultimate less incurred |

In [6]:
premium = df[df.line_of_business == 'Commercial Property'] \
            .groupby('accident_year')['earned_premium'].first().to_dict()

result = project_reserves(paid, incurred, premium, initial_expected_loss_ratio=0.75,
                          tail_factor=tail)
result[['accident_year', 'latest_paid_claims', 'case_reserves', 'cdf_to_ultimate',
        'cl_ultimate_claims', 'cl_total_unpaid_reserve', 'cl_ibnr_reserve']]

,accident_year,latest_paid_claims,case_reserves,cdf_to_ultimate,cl_ultimate_claims,cl_total_unpaid_reserve,cl_ibnr_reserve
0,2019,3750000.0,50000.0,1.013333,3800000.00,50000.00,0.00
1,2020,3900000.0,150000.0,1.070423,4174647.89,274647.89,124647.89
2,2021,3800000.0,350000.0,1.217503,4626513.28,826513.28,476513.28
3,2022,3050000.0,650000.0,1.647211,5023992.36,1973992.36,1323992.36
4,2023,1850000.0,1100000.0,2.987314,5526531.19,3676531.19,2576531.19


In [7]:
unpaid = result['cl_total_unpaid_reserve'].sum()
case = result['case_reserves'].sum()
ibnr = result['cl_ibnr_reserve'].sum()
print(f"Total unpaid   : GBP {unpaid:,.0f}")
print(f"  case reserves: GBP {case:,.0f}")
print(f"  IBNR         : GBP {ibnr:,.0f}")
print(f"\ncloses: {abs(unpaid - case - ibnr) < 0.01}")

no_tail = project_reserves(paid, incurred, premium, 0.75, tail_factor=1.0)
row = no_tail[no_tail.accident_year == 2020].iloc[0]
print(f"\nAY2020 with no tail, the old published basis:")
print(f"  total unpaid reported as IBNR: GBP {row['cl_total_unpaid_reserve']:,.0f}")
print(f"  true IBNR                    : GBP {row['cl_ibnr_reserve']:,.0f}")
print(f"  overstatement                : {row['cl_total_unpaid_reserve']/row['cl_ibnr_reserve']:.2f}x")

Total unpaid   : GBP 6,801,685
  case reserves: GBP 2,300,000
  IBNR         : GBP 4,501,685

closes: True

AY2020 with no tail, the old published basis:
  total unpaid reported as IBNR: GBP 219,718
  true IBNR                    : GBP 69,718
  overstatement                : 3.15x


## 5. Uncertainty: Mack (1993)

A point estimate is not a reserving deliverable. Mack gives a distribution-free
standard error of the chain-ladder reserve.

The implementation is validated against the Taylor & Ashe (1983) triangle, for which
per-accident-year standard errors are published.

In [8]:
from tests.test_reserving import taylor_ashe_triangle, PUBLISHED_MACK_SE

benchmark = mack_standard_error(taylor_ashe_triangle(), tail_factor=1.0)
print(f"{'AY':>3} {'computed':>12} {'published':>12}  match")
for ay, expected in PUBLISHED_MACK_SE.items():
    ok = abs(benchmark[ay] - expected) < 1.0
    print(f"{ay:>3} {benchmark[ay]:>12,.0f} {expected:>12,.0f}  {ok}")

 AY     computed    published  match
  1            0            0  True
  2       75,535       75,535  True
  3      121,699      121,699  True
  4      133,549      133,549  True
  5      261,406      261,406  True
  6      411,010      411,010  True
  7      558,317      558,317  True
  8      875,328      875,328  True
  9      971,258      971,258  True
 10    1,363,155    1,363,155  True


In [9]:
result[['accident_year', 'cl_total_unpaid_reserve', 'cl_reserve_standard_error',
        'cl_reserve_cv_pct', 'cl_reserve_75th_percentile', 'cl_reserve_95th_percentile']]

,accident_year,cl_total_unpaid_reserve,cl_reserve_standard_error,cl_reserve_cv_pct,cl_reserve_75th_percentile,cl_reserve_95th_percentile
0,2019,50000.00,0.00,0.00,50000.00,50000.00
1,2020,274647.89,21461.34,7.81,289123.56,309949.64
2,2021,826513.28,54549.74,6.60,863307.08,916242.15
3,2022,1973992.36,60736.84,3.08,2014959.36,2073898.39
4,2023,3676531.19,88988.19,2.42,3736553.73,3822907.87


**Read these intervals with caution.** Coefficients of variation land between 1% and
8%. A real reserving triangle produces materially wider ranges. This dataset is
close to deterministic, so Mack's process variance parameters are near zero and the
uncertainty is understated by construction. The method is correct; the data is too
well behaved to exercise it.

## 6. The finding

The projected ultimate loss ratio deteriorates monotonically across every accident
year, and most of the held reserve sits in the least mature year where least is known.

In [10]:
summary = result[['accident_year', 'cl_loss_ratio', 'bf_loss_ratio']].copy()
summary['cl_loss_ratio'] = (summary['cl_loss_ratio'] * 100).round(1)
summary['bf_loss_ratio'] = (summary['bf_loss_ratio'] * 100).round(1)
summary['gap_pts'] = (summary['cl_loss_ratio'] - summary['bf_loss_ratio']).round(1)
print(summary.to_string(index=False))

first, last = result['cl_loss_ratio'].iloc[0], result['cl_loss_ratio'].iloc[-1]
print(f"\nDeterioration 2019 to 2023: {(last-first)*100:+.1f} points")
share = result['cl_total_unpaid_reserve'].iloc[-1] / result['cl_total_unpaid_reserve'].sum()
print(f"Share of total unpaid sitting in AY2023: {share:.1%}")

 accident_year  cl_loss_ratio  bf_loss_ratio  gap_pts
          2019           76.0           76.0      0.0
          2020           79.5           79.2      0.3
          2021           84.1           82.5      1.6
          2022           86.6           82.0      4.6
          2023           90.6           80.2     10.4

Deterioration 2019 to 2023: +14.6 points
Share of total unpaid sitting in AY2023: 54.1%
